# FinQA 数据资产准备

本 Notebook 为银行金融大模型量化项目准备三类资产：

- `calibration`：用于 AWQ/GPTQ/SmoothQuant 校准；
- `quality_dev` 和 `quality_test`：用于质量评估；
- `high_risk_regression`：固定的金融数值回归集。

数据会缓存到 Google Drive，避免 Colab 运行时重置后重复下载。

In [19]:
!pip -q install -U datasets huggingface_hub pandas pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 载入项目源码

如果你已经把项目目录上传到 Colab，请修改 `PROJECT_DIR`。如果项目放在 Git 仓库，可以填写 `REPO_URL` 自动克隆。

In [21]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = '/content/drive/MyDrive/OptimizationSystem'
REPO_URL = ''

if REPO_URL and not Path(PROJECT_DIR).exists():
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)

if not Path(PROJECT_DIR).exists():
    raise FileNotFoundError(
        f'{PROJECT_DIR} 不存在，请上传项目或填写 REPO_URL。'
    )

sys.path.insert(0, PROJECT_DIR)

In [22]:
import importlib
import src.data.finqa_assets as finqa_assets

importlib.reload(finqa_assets)
prepare_assets = finqa_assets.prepare_assets
print('FinQA loader:', finqa_assets.__file__)

OUTPUT_DIR = (
    '/content/drive/MyDrive/'
    'banking_llm_project/datasets'
)

assets = prepare_assets(
    output_dir=OUTPUT_DIR,
    calibration_size=1024,
    seed=42,
)

assets.summary()

FinQA loader: /content/drive/MyDrive/OptimizationSystem/src/data/finqa_assets.py


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

0000.parquet:   0%|          | 0.00/155k [00:00<?, ?B/s]

...

Saving the dataset (0/1 shards):   0%|          | 0/1024 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/883 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1064 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/91 [00:00<?, ? examples/s]

{'calibration': 1024,
 'quality_dev': 883,
 'quality_test': 1064,
 'high_risk_regression': 91,
 'manifest': {'finqa_dataset': 'czyssrs/FinQA',
  'verified_dataset': 'Aiera/finqa-verified',
  'finqa_revision': 'main',
  'verified_revision': 'default',
  'seed': 42,
  'calibration_size': 1024,
  'sizes': {'calibration': 1024,
   'quality_dev': 883,
   'quality_test': 1064,
   'high_risk_regression': 91},
  'hashes': {'calibration': '89dbd2dcd40a790cc5cb94a3728de2f9b0c070ab9b8a7f9eeae42c3ced851e9a',
   'quality_dev': '7ba8d507634daa266268414448444421a18868e926632d805949058118b8a6d8',
   'quality_test': '0a89c300bd7e6007f5f210933d115f4757b9a54c20aeba7520bb5d1ad2f03d97',
   'high_risk_regression': '81a6ba140d9806e36d6be5b8f4362a506ff710c853e8f3448889190942a31626'}}}

In [23]:
print('校准集:', len(assets.calibration))
print('质量调优集:', len(assets.quality_dev))
print('最终质量评估集:', len(assets.quality_test))
print('高风险回归集:', len(assets.high_risk_regression))
print('第一条校准 Prompt:')
print(assets.calibration_prompts[0][:2000])

校准集: 1024
质量调优集: 883
最终质量评估集: 1064
高风险回归集: 91
第一条校准 Prompt:
You are a financial analysis assistant.

Financial context:
contractual cash flows following is a summary of our contractual payment obligations related to our consolidated debt , contingent consideration , operating leases , other commitments and long-term liabilities at september 30 , 2011 ( see notes 9 and 13 to the consolidated financial statements contained this annual report ) , ( in thousands ) : .

Financial table:
obligation | payments due by period total | payments due by period less than 1year | payments due by period 1-3 years | payments due by period 3-5 years | payments due by period thereafter
short-term debt obligations | $ 26677 | $ 26677 | $ 2014 | $ 2014 | $ 2014
cash premium on convertible notes due march 2012 ( 1 ) | 23558 | 23558 | 2014 | 2014 | 2014
other commitments ( 2 ) | 5170 | 3398 | 1772 | 2014 | 2014
operating lease obligations | 37788 | 8247 | 13819 | 9780 | 5942
contingent consideration for busi

## 接入量化算法

`assets.calibration_prompts` 可以直接传给你的 AWQ/GPTQ 校准流程。不要把答案字段拼进校准 Prompt。

In [ ]:
calibration_prompts = assets.calibration_prompts

for prompt in calibration_prompts[:3]:
    print(prompt[:500])
    print('---')

In [ ]:
from src.data.finqa_assets import load_assets

reloaded_assets = load_assets(OUTPUT_DIR)
reloaded_assets.summary()

## 使用规则

1. 用 `calibration` 计算量化参数。
2. 用 `quality_dev` 选择 group size、敏感层和混合精度。
3. 配置冻结后才使用 `quality_test` 生成最终报告。
4. 每个 FP16、INT8、INT4、剪枝或后端版本都运行 `high_risk_regression`。
5. 重点统计“FP16 正确但压缩模型错误”的新增退化。